# Structure-Validated De Novo Binder Design

End-to-end pipeline: **BoltzGen -> LigandMPNN -> BoltzFold**

1. **BoltzGen** generates de novo backbones conditioned on a target
2. **LigandMPNN** designs sequences for each backbone
3. **BoltzFold** refolds each designed sequence to validate structure and binding

In [1]:
import torch

from evedesign.system import System, Protein
from evedesign.models.boltzgen import BoltzGenGenerator
from evedesign.models.mpnn import LigandMPNN
from evedesign.models.boltzfold import BoltzFoldTransformer

[14:15:53] Initializing Normalizer


## Step 1: Define target and binder

This notebook uses **1G13** (human GM2 activator protein, chain A) as the target - the canonical test case from BoltzGen's `vanilla_protein/1g13prot.yaml` example.

System layout:

- **Target** (`rep=<1G13 chain A sequence>` plus `structures`): the crystal structure is attached, so BoltzGen receives a `file:` entry and designs against the experimental conformation. Without `structures` it would auto-generate an MSA and fold the target itself, which is the right choice only when no structure is available.
- **Binder** (`rep=None`, `min_length=80`, `max_length=140`): the de novo binder. No starting sequence, no starting backbone - BoltzGen invents both. The length range matches BoltzGen's vanilla binder convention.

To swap in a different target, change `TARGET_PDB_ID` and `TARGET_CHAIN` below.

In [2]:
import biotite.structure as struc

from evedesign.structure import Structure, StructureFile
from evedesign.tools.mmseqs2 import add_sequences_mmseqs2

TARGET_PDB_ID = "1G13"           # GM2 activator protein
TARGET_CHAIN = "A"

NUM_BACKBONES = 5                # backbones from BoltzGen
NUM_SEQUENCES_PER_BACKBONE = 4   # MPNN sequences per backbone

chain = StructureFile.from_id(TARGET_PDB_ID).get_model().get_chain(TARGET_CHAIN)
target = Structure(chain.atom_array[struc.filter_amino_acids(chain.atom_array)])

system = System([
    Protein(
        rep="".join(target.res_df().res_name_oneletter),
        id="target",
        structures={TARGET_PDB_ID.lower(): target},
    ),
    Protein(rep=None, min_length=80, max_length=140, id="binder"),
])

system = add_sequences_mmseqs2(system)

2026-08-18 14:15:54.879 | WARNING  | evedesign.tools.api_utils:_request_with_retries:37 - Error while contacting %s. Retrying... (%d/%d)
2026-08-18 14:15:54.880 | WARNING  | evedesign.tools.api_utils:_request_with_retries:43 - Error: %s
2026-08-18 14:16:02.698 | WARNING  | evedesign.tools.api_utils:_request_with_retries:37 - Error while contacting %s. Retrying... (%d/%d)
2026-08-18 14:16:02.698 | WARNING  | evedesign.tools.api_utils:_request_with_retries:43 - Error: %s
2026-08-18 14:16:07.917 | WARNING  | evedesign.tools.api_utils:_request_with_retries:37 - Error while contacting %s. Retrying... (%d/%d)
2026-08-18 14:16:07.919 | WARNING  | evedesign.tools.api_utils:_request_with_retries:43 - Error: %s
2026-08-18 14:16:13.132 | WARNING  | evedesign.tools.api_utils:_request_with_retries:37 - Error while contacting %s. Retrying... (%d/%d)
2026-08-18 14:16:13.133 | WARNING  | evedesign.tools.api_utils:_request_with_retries:43 - Error: %s
2026-08-18 14:16:18.352 | WARNING  | evedesign.tools

HTTPError: 429 Client Error: Too Many Requests for url: https://api.colabfold.com/ticket/msa

## Step 2: Generate backbones with BoltzGen

Run BoltzGen's diffusion model to produce candidate binder backbones. The pipeline:
1. Writes a BoltzGen YAML spec for the system
2. Invokes the `boltzgen run` CLI
3. Parses outputs into `SystemInstance` objects with structures on `EntityInstance.models["model_0"]`

In [ ]:
generator = BoltzGenGenerator(
    protocol="protein-anything",
    device="cuda",
    num_devices=torch.cuda.device_count(),
    skip_inverse_folding=True,
    budget=10,
).build(system)

# entities=[1]: design the binder, hold the 1G13 target fixed
backbones = generator.generate(num_designs=NUM_BACKBONES, entities=[1])

for i, bb in enumerate(backbones):
    print(f"Backbone {i}: score={bb.score:.3f}, "
          f"confidence={bb.confidence:.3f}, "
          f"binder length={len(bb[1].rep)}")

2026-08-12 16:36:26.591 | INFO     | evedesign.models.boltz.convert_design:system_to_boltzgen_yaml:643 - System has 2 entities: 1 designed, 1 context
2026-08-12 16:36:26.595 | INFO     | evedesign.models.boltzgen:generate:309 - BoltzGen YAML written to /tmp/boltzgen_fdpopit1/design_spec.yaml
2026-08-12 16:36:26.595 | INFO     | evedesign.models.boltzgen:generate:321 - Running BoltzGen: boltzgen run /tmp/boltzgen_fdpopit1/design_spec.yaml --output /tmp/boltzgen_fdpopit1/output --protocol protein-anything --num_designs 5 --num_workers 1 --inverse_fold_num_sequences 1 --budget 10 --design_checkpoints huggingface:boltzgen/boltzgen-1:boltzgen1_diverse.ckpt huggingface:boltzgen/boltzgen-1:boltzgen1_adherence.ckpt --inverse_fold_checkpoint huggingface:boltzgen/boltzgen-1:boltzgen1_ifold.ckpt --folding_checkpoint huggingface:boltzgen/boltzgen-1:boltz2_conf_final.ckpt --devices 2 --skip_inverse_folding


## Step 3: Design sequences with LigandMPNN and refold with BoltzFold

For each BoltzGen backbone: promote it onto `Entity.structures` so LigandMPNN can design sequences against it, then refold those sequences with BoltzFold to check they fold back into the intended structure and keep a confident interface.

Results are kept as `(backbone, refolds)` pairs, so each design stays attached to the backbone it came from.

In [ ]:
results = []   # (backbone, refolded designs) per BoltzGen backbone

for bb_idx, backbone in enumerate(backbones):
    print(f"--- Backbone {bb_idx} ---")

    bb_system = system.apply_instance(backbone)

    seqs = LigandMPNN(
        model_name="ligandmpnn_v_32_010_25",
        device="cuda",
    ).build(bb_system).generate(
        num_designs=NUM_SEQUENCES_PER_BACKBONE,
        temperature=0.1,
        entities=[1],
    )

    # BoltzFold needs a defined binder sequence; all designs from one
    # backbone share its length, so any of them works as the template
    fold_system = system.apply_instance(seqs[0])
    fold_system[0].sequences = system[0].sequences   # apply_instance drops MSAs

    refolds = BoltzFoldTransformer(
        device="cuda",
        sampling_steps=200,
        diffusion_samples=3,
        recycling_steps=3,
        use_msa=True,
        use_kernels=False,
    ).build(fold_system).transform(seqs)

    results.append((backbone, refolds))

    for r in refolds:
        print(f"  score={r.score:.3f}, confidence={r.confidence:.3f}, "
              f"{''.join(r[1].rep[:30])}...")

## Step 4: Filter and rank

Filter the refolded designs by structural confidence:
- `confidence > 0.7` (complex pLDDT - overall structure confidence)
- `score > 0.5` (the configured score_attribute, by default `confidence_score`)

Then sort by score descending. In production you'd also filter by interface metrics (ipTM, ipSAE) and liability counts.

In [ ]:
import pandas as pd

PLDDT_THRESHOLD = 0.7
SCORE_THRESHOLD = 0.5

df = pd.DataFrame([
    {
        "backbone": bb_idx,
        "design": i,
        "score": r.score,
        "confidence": r.confidence,
        "length": len(r[1].rep),
        "sequence": "".join(r[1].rep),
    }
    for bb_idx, (_, refolds) in enumerate(results)
    for i, r in enumerate(refolds)
])

assert not df.empty, "No refolded designs - check the BoltzGen and MPNN steps"

passing = df[
    (df["confidence"] > PLDDT_THRESHOLD)
    & (df["score"] > SCORE_THRESHOLD)
].sort_values("score", ascending=False)

print(f"{len(passing)}/{len(df)} designs passed "
      f"(confidence > {PLDDT_THRESHOLD}, score > {SCORE_THRESHOLD})")
print(passing.to_string(index=False) if len(passing) else
      "None, try lowering thresholds or increasing NUM_BACKBONES.")